# 03 – Feature Engineering

### Purpose
This notebook creates engineered features that enhance the predictive power and interpretability of the TED CAN dataset.
All features are based on insights from Notebook 01 (EDA) and domain knowledge of public procurement.

### Steps
- Load cleaned‑saved datasets
- Apply feature engineering pypline
- Save full datasets

--------------------
#### Imports & Setup & Dataset
-------------------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.feature_engineering import feature_engineering
from my_scripts.eda import overview

In [5]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset_clean.pkl")
df_de = pd.read_pickle("../data/dataset_de_clean.pkl")

print("EU dataset:", df.shape)
print("DE dataset:", df_de.shape)

EU dataset: (4039906, 34)
DE dataset: (303349, 34)


---------------------------------------------------------
### Apply feature engineering

-----------------------------

In [6]:
# ---------------------------------------------------------
# full EU dataset
# ---------------------------------------------------------

df_fe = feature_engineering(df)


In [7]:
# shape of the EU dataset
print(df_fe.shape)

(4039906, 43)


In [8]:
# inspekt of the EU dataset
overview(df_fe)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int64,4039906,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ID_TYPE,int64,4039906,0,0.00,7,"[3, 6, 18, 25, 21, 23, 22]"
XSD_VERSION,str,4039906,0,0.00,7,"[D205, D206, D207, R207.S3, R208.S1, R208.S2, ..."
CANCELLED,int64,4039906,0,0.00,2,"[0, 1]"
CORRECTIONS,int64,4039906,0,0.00,10,"[0, 1, 2, 3, 5, 4, 84, 7, 8, 6]"
ISO_COUNTRY_CODE,str,4039906,0,0.00,33,"[DE, FR, ES, SE, PL, IT, HU, CY, UK, RO, PT, N..."
CAE_TYPE,str,4039906,0,0.00,10,"[8, 3, 1, 6, R, N, 4, 5A, 5, Z]"
B_AWARDED_BY_CENTRAL_BODY,str,4039906,0,0.00,3,"[Unknown, Y, N]"
TYPE_OF_CONTRACT,str,4039906,0,0.00,3,"[W, U, S]"
TAL_LOCATION_NUTS,str,4039906,0,0.00,9310,"[DED31, DE913, Unknown, DE13A, ES3, FR91, FR52..."


In [9]:
# ---------------------------------------------------------
# dataset for Germany
# ---------------------------------------------------------

df_de_fe = feature_engineering(df_de)


In [10]:
# shape of the EU dataset
print(df_de_fe.shape)

(303349, 43)


In [11]:
# inspekt of the EU dataset
overview(df_de_fe)

,dtype,total,missing_n,missing_%,uniques_n,uniques
YEAR,int64,303349,0,0.00,9,"[2008, 2009, 2010, 2011, 2012, 2013, 2014, 201..."
ID_TYPE,int64,303349,0,0.00,5,"[3, 6, 18, 25, 21]"
XSD_VERSION,str,303349,0,0.00,7,"[D205, D206, D207, R207.S3, R208.S1, R208.S2, ..."
CANCELLED,int64,303349,0,0.00,2,"[0, 1]"
CORRECTIONS,int64,303349,0,0.00,3,"[0, 1, 2]"
ISO_COUNTRY_CODE,str,303349,0,0.00,1,[DE]
CAE_TYPE,str,303349,0,0.00,10,"[8, 3, 6, N, 1, R, 4, 5A, 5, Z]"
B_AWARDED_BY_CENTRAL_BODY,str,303349,0,0.00,3,"[Unknown, Y, N]"
TYPE_OF_CONTRACT,str,303349,0,0.00,3,"[W, U, S]"
TAL_LOCATION_NUTS,str,303349,0,0.00,2107,"[DED31, DE913, DE13A, DE915, DE712, DEA52, DE9..."


#### Notes: Feature Engineering Summary
Feature engineering enriches the cleaned dataset with additional analytical variables that improve modelling of tender competition, risk, and procedural behaviour.

1. Dataset Size After Features Engineering
- Full EU dataset:
  - Before: 4,039,906 rows × 34 columns
  - After: 4,039,906 rows × 43 columns

- Germany-only dataset:
  - Before: 303,349 rows × 34 columns
  - After: 303,349 rows × 43 columns

2. Created Features
- Date‑based features
  - AWARD_YEAR
  - AWARD_MONTH
  - AWARD_QUARTER
  - DAYS_TO_AWARD (time between dispatch and award)
These variables capture temporal patterns, seasonality, and procedural duration.

- CPV hierarchy features
  - CPV_DIVISION (first 2 digits)
  - CPV_GROUP (first 3 digits)
  - CPV_CLASS (first 4 digits)
The CPV field was converted from float to string and cleaned (removal of “.0”) to enable correct hierarchical extraction.

- Competition features
  - IS_FAILED_TENDER (≤1 offer)
  - IS_LOW_COMPETITION (=2 offers)
  - OFFERS_BIN (failed / low / medium / high)
These features form the basis for competition and risk modelling.

- Value‑based features
  - VALUE_BIN (micro / small / medium / large / mega)
This categorisation helps the model handle highly skewed contract value distributions.

- Tender complexity features
  - HAS_MULTIPLE_LOTS
  - LOTS_BIN (single / few / many / mega)
These variables reflect structural complexity of the tender.

- Boolean indicators (Y/N into 1/0)
  - B_ELECTRONIC_AUCTION
  - B_DYN_PURCH_SYST
  - B_ACCELERATED
  - B_AWARDED_TO_A_GROUP
  - B_CONTRACTOR_SME
All Y/N fields were mapped to binary numeric format.

- Missing‑information flags
  - VALUE_EURO_MISSING
  - AWARD_VALUE_EURO_MISSING
  - NUMBER_OFFERS_MISSING
These indicators allow the model to capture informative missingness.

3. Removed Raw Columns
After generating engineered features, the following original fields were dropped to avoid redundancy:
  - CPV
  - DT_AWARD, DT_DISPATCH
  - VALUE_EURO, AWARD_VALUE_EURO, AWARD_EST_VALUE_EURO
  - LOTS_NUMBER
  - NUMBER_OFFERS



------------------
### Save engineered datasets

-------------------

In [12]:
df_fe.to_pickle("../data/dataset_fe.pkl")
df_de_fe.to_pickle("../data/dataset_de_fe.pkl")